In [29]:
import pandas as pd
import numpy as np
import json

In [30]:
PATH_DATA = "data/"

In [3]:
df_icd = pd.read_csv(PATH_DATA +"LIBCIM10MULTI.TXT", sep="|",header=None,names=["code","aut_mco","pos","aut_ssr","lib_court","libelle"],encoding="latin-1")
df_icd.code = df_icd.code.str.replace(" ","")
df_index_icd = pd.read_csv(PATH_DATA + "cim_index_modifie.csv", sep=";")

In [4]:
df_icd_chap20 = pd.read_csv(PATH_DATA +"LIBCIM10MULTI_ch20.TXT", sep="|",header=None,names=["code","aut_mco","pos","aut_ssr","lib_court","libelle"],encoding="latin-1")
df_icd_chap20.code = df_icd_chap20.code.str.replace(" ","")

cat_motif = ["Z"+ str(x).zfill(2) for x in range(0,55)]
cat_facteurs = ["Z"+ str(x).zfill(2) for x in range(55,100)]
cat_sympt = ["R"+ str(x).zfill(2) for x in range(0,100)]

In [5]:
df_icd= df_icd.assign(groupe = np.where(df_icd.code.isin(df_icd_chap20.code),"Causes externes",
                             np.where(df_icd.code.str.slice(0,3).isin(cat_motif),"Motifs de recours",
                             np.where(df_icd.code.str.slice(0,3).isin(cat_facteurs),"Facteurs influents",
                             np.where(df_icd.code.str.slice(0,3).isin(cat_sympt),"Symptomes",
                             "Diagnostics")))))

In [6]:
df_index_icd = df_index_icd.assign(groupe = np.where(df_index_icd.code.isin(df_icd_chap20.code),"Causes externes",
                             np.where(df_index_icd.code.str.slice(0,3).isin(cat_motif),"Motifs de recours",
                             np.where(df_index_icd.code.str.slice(0,3).isin(cat_facteurs),"Facteurs influents",
                             np.where(df_index_icd.code.str.slice(0,3).isin(cat_sympt),"Symptomes",
                             "Diagnostics")))))

In [7]:
df_index_icd[df_index_icd.code.str.slice(-1) == "9" ].drop_duplicates("code").groupby("groupe").size()

groupe
Diagnostics           914
Facteurs influents     17
Motifs de recours      35
Symptomes              19
dtype: int64

In [48]:
df_index_icd[(df_index_icd.code.str.slice(-1) == "9") & ( df_index_icd.groupe=="Facteurs influents")  ]

,code,icd_description,index_orginal,index_reformulate,groupe
275362,Z579,Difficultés liées à l'exposition professionnel...,"Difficulté(s) de(s), liées à, exposition à, fa...",difficultés liées à l'exposition à un facteur ...,Facteurs influents
275363,Z579,Difficultés liées à l'exposition professionnel...,"Difficulté(s) de(s), liées à, exposition à, fa...",difficultés liées à l'exposition professionnel...,Facteurs influents
275364,Z579,Difficultés liées à l'exposition professionnel...,"Difficulté(s) de(s), liées à, exposition à, fa...",difficultés d'exposition à un facteur de risqu...,Facteurs influents
275365,Z579,Difficultés liées à l'exposition professionnel...,"Difficulté(s) de(s), liées à, exposition à, fa...",difficultés d'exposition professionnelle à un ...,Facteurs influents
275366,Z579,Difficultés liées à l'exposition professionnel...,"Difficulté(s) de(s), liées à, exposition à, fa...",difficulté liée à l'exposition à un facteur de...,Facteurs influents
...,...,...,...,...,...
280607,Z899,"Absence acquise de membre, sans précision","Moignon (chirurgical) d'amputation, guéri ou a...",amputation avec moignon guéri,Facteurs influents
280608,Z899,"Absence acquise de membre, sans précision","Moignon (chirurgical) d'amputation, guéri ou a...",amputation avec moignon ancien,Facteurs influents
280609,Z899,"Absence acquise de membre, sans précision","Moignon (chirurgical) d'amputation, guéri ou a...",amputation avec moignon chirurgical,Facteurs influents
280610,Z899,"Absence acquise de membre, sans précision","Moignon (chirurgical) d'amputation, guéri ou a...",amputation avec moignon chirurgical guéri,Facteurs influents


In [52]:
df_index_icd[(df_index_icd.code.str.slice(-1) == "9") & ( df_index_icd.groupe=="Diagnostics")  ].to_excel("data/Controle_index_sai.xlsx",index=False)

In [37]:
df_icd[df_icd.code.str.slice(-1) == "9" ].drop_duplicates("code").groupby("groupe").size()

groupe
Causes externes       3690
Diagnostics           1539
Facteurs influents      35
Motifs de recours       38
Symptomes               35
dtype: int64

In [10]:
df_icd_chapitres = pd.read_excel(PATH_DATA +"icd_chapters.xlsx")


In [17]:
df_icd = df_icd.assign(categorie = df_icd.code.str.slice(0,3))

In [19]:
df_icd.merge(df_icd_chapitres,left_on="categorie",right_on="diag_deb",how="left").ffill()

,code,aut_mco,pos,aut_ssr,lib_court,libelle,categorie,hiera,diag_deb,diag_fin,lib,annee_deb,annee_fin,en_cours,niveau_fin
0,A00,3,NNN,3,CHOLERA,Choléra,A00,1.0,A00,B99,Certaines maladies infectieuses et parasitaires,2009.0,2025.0,1.0,0.0
1,A000,0,OOO,0,"CHOLERA A VIBRIO CHOLERAE 01, BIOVAR CHOLERAE","Choléra à Vibrio cholerae 01, biovar cholerae",A00,1.0,A00,B99,Certaines maladies infectieuses et parasitaires,2009.0,2025.0,1.0,0.0
2,A001,0,OOO,0,"CHOLERA A VIBRIO CHOLERAE 01, BIOVAR EL TOR","Choléra à Vibrio cholerae 01, biovar El Tor",A00,1.0,A00,B99,Certaines maladies infectieuses et parasitaires,2009.0,2025.0,1.0,0.0
3,A009,0,OOO,0,"CHOLERA, SAI","Choléra, sans précision",A00,1.0,A00,B99,Certaines maladies infectieuses et parasitaires,2009.0,2025.0,1.0,0.0
4,A01,3,NNN,3,FIEVRES TYPHOIDE ET PARATYPHOIDE,Fièvres typhoïde et paratyphoïde,A01,1.0,A00,B99,Certaines maladies infectieuses et parasitaires,2009.0,2025.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42892,Z992+8,0,ONO,0,"DEPENDANCE ENVERS UNE DIALYSE RENALE, NCA","Dépendance envers une dialyse rénale, autre",Z99,21.0,Z00,Z80,Facteurs influant sur l'état de santé et motif...,2009.0,2025.0,1.0,0.0
42893,Z993,0,ONO,0,DEPENDANCE ENVERS UN FAUTEUIL ROULANT,Dépendance envers un fauteuil roulant,Z99,21.0,Z00,Z80,Facteurs influant sur l'état de santé et motif...,2009.0,2025.0,1.0,0.0
42894,Z994,0,ONO,0,DEPENDANCE ENVERS UN COEUR ARTIFICIEL,Dépendance envers un coeur artificiel,Z99,21.0,Z00,Z80,Facteurs influant sur l'état de santé et motif...,2009.0,2025.0,1.0,0.0
42895,Z998,0,ONO,0,DEPENDANCE ENVERS D'AUTRES MACHINES ET APP. AU...,Dépendance envers d'autres machines et apparei...,Z99,21.0,Z00,Z80,Facteurs influant sur l'état de santé et motif...,2009.0,2025.0,1.0,0.0


In [53]:
df_icd[df_icd.code.str.contains('N02')]

,code,aut_mco,pos,aut_ssr,lib_court,libelle,categorie,groupe
10445,N02,3,NNN,3,"HEMATURIE RECID., PERSIST.",Hématurie récidivante et persistante,N02,Diagnostics
10446,N020,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC ANOM. GLOM. MI...",Hématurie récidivante et persistante avec anom...,N02,Diagnostics
10447,N0200,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC ANOM. GLOM. MI...",Hématurie récidivante et persistante avec anom...,N02,Diagnostics
10448,N0209,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC ANOM. GLOM. MI...",Hématurie récidivante et persistante avec anom...,N02,Diagnostics
10449,N021,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC LES. GLOM. SEG...",Hématurie récidivante et persistante avec lési...,N02,Diagnostics
10450,N0210,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC LES. GLOM. SEG...",Hématurie récidivante et persistante avec lési...,N02,Diagnostics
10451,N0219,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC LES. GLOM. SEG...",Hématurie récidivante et persistante avec lési...,N02,Diagnostics
10452,N022,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC G.N. MEMBRANEU...",Hématurie récidivante et persistante avec glom...,N02,Diagnostics
10453,N023,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC G.N. PROLIF. M...",Hématurie récidivante et persistante avec glom...,N02,Diagnostics
10454,N024,0,OOO,0,"HEMATURIE RECID., PERSIST. AVEC G.N. PROLIF. E...",Hématurie récidivante et persistante avec glom...,N02,Diagnostics


In [55]:
df_index_icd[df_index_icd.index_reformulate=="hématurie"]

,code,icd_description,index_orginal,index_reformulate,groupe
144982,N029,"Hématurie récidivante et persistante, sans pré...","Hématurie (essentielle), intermittente (voir a...",hématurie,Diagnostics
144986,N029,"Hématurie récidivante et persistante, sans pré...","Hématurie (essentielle), paroxystique (voir au...",hématurie,Diagnostics
228862,R31,"Hématurie, sans précision",Hématurie (essentielle),hématurie,Symptomes
228864,R31,"Hématurie, sans précision","Hématurie (essentielle), due aux sulfamides, m...",hématurie,Symptomes
228880,R31,Hématurie,"Hémorragie (de) (due à), urinaire nca",hématurie,Symptomes


In [68]:
df_tmp = df_index_icd.drop_duplicates(["code","index_reformulate"]).groupby("index_reformulate").size().to_frame('nb').reset_index()

In [69]:
df_tmp = df_tmp[df_tmp.nb>1]

In [73]:
df_index_icd[df_index_icd.index_reformulate.isin(df_tmp.index_reformulate)].sort_values("index_reformulate").to_excel("data/duplicates_index_entries.xlsx")

In [ ]:
import requests
from lxml import etree
import pandas as pd

import os
import requests

# 1. Télécharger le XSD depuis l'URL
xsd_url = "https://www.orphacode.org/data/xsd_jpg/ORPHA_ICD10_mapping_en_2020.xsd"
response = requests.get(xsd_url)

# Chemin du dossier de destination
dossier_destination = "data/Orphanet_Nomenclature_Pack_FR_2025/"

# Créer le dossier s'il n'existe pas
os.makedirs(dossier_destination, exist_ok=True)

# Chemin complet pour le fichier XSD
chemin_xsd = os.path.join(dossier_destination, "ORPHA_ICD10_mapping_en_2020.xsd")

# 2. Écrire le fichier XSD dans le dossier spécifié
with open(chemin_xsd, 'wb') as f:
    f.write(response.content)

In [20]:


# 3. Charger le schéma XSD
with open(chemin_xsd, 'rb') as f:
    xsd_schema = etree.XMLSchema(file=f)

# 4. Parser le fichier XML avec validation
xml_file = os.path.join(dossier_destination, "ORPHA_ICD10_mapping_fr_2025.xml")

with open(xml_file, 'rb') as f:
    xml_doc = etree.parse(f)
    xsd_schema.assertValid(xml_doc)  # Valide le XML selon le XSD




In [23]:
# Liste pour stocker les données
data = []

# Parcourir chaque élément Disorder
for disorder in xml_doc.xpath('//Disorder'):
    orpha_code = disorder.xpath('OrphaCode/text()')[0]
    name = disorder.xpath('Name/text()')[0]
    name_lang = disorder.xpath('Name/@lang')[0]

    # Extraire les synonymes
    synonyms = disorder.xpath('SynonymList/Synonym/text()')
    synonyms_lang = disorder.xpath('SynonymList/Synonym/@lang')
    synonyms_list = ', '.join([f"{synonym} ({lang})" for synonym, lang in zip(synonyms, synonyms_lang)]) if synonyms else None

    # Extraire les ExternalReferences
    for external_ref in disorder.xpath('ExternalReferenceList/ExternalReference'):
        source = external_ref.xpath('Source/text()')[0]
        reference = external_ref.xpath('Reference/text()')[0]

        # Extraire les relations ICD
        mapping_icd_relation_name = external_ref.xpath('DisorderMappingICDRelation/Name/text()')[0]
        mapping_icd_relation_id = external_ref.xpath('DisorderMappingICDRelation/@id')[0]

        # Ajouter les données à la liste
        data.append({
            'OrphaCode': orpha_code,
            'Name': name,
            'Name_lang': name_lang,
            'SynonymList': synonyms_list,
            'Source': source,
            'Reference': reference,
            'DisorderMappingICDRelation_name': mapping_icd_relation_name,
            'DisorderMappingICDRelation_id': mapping_icd_relation_id,
        })

# Créer le DataFrame
df = pd.DataFrame(data)



In [24]:
df

,OrphaCode,Name,Name_lang,SynonymList,Source,Reference,DisorderMappingICDRelation_name,DisorderMappingICDRelation_id
0,166024,Syndrome de dysplasie épiphysaire multiple-mac...,fr,Dysplasie épiphysaire multiple type Al-Gazali ...,ICD-10,Q77.3,Code attribué (CIM-10/CIM-11: Le code cible es...,21604
1,58,Maladie d'Alexander,fr,AxD (fr),ICD-10,G93.8,Code attribué (CIM-10/CIM-11: Le code cible es...,21604
2,166032,Syndrome de dysplasie épiphysaire multiple-min...,fr,None,ICD-10,Q77.3,Code attribué (CIM-10/CIM-11: Le code cible es...,21604
3,61,Alpha-mannosidose,fr,Déficit en alpha-D-mannosidase lysosomale (fr),ICD-10,E77.1,Terme d'inclusion (CIM-10: L'entité Orphanet e...,21590
4,166029,Syndrome de dysplasie épiphysaire multiple-dys...,fr,None,ICD-10,Q77.3,Code attribué (CIM-10/CIM-11: Le code cible es...,21604
...,...,...,...,...,...,...,...,...
8328,619979,Syndrome de retard du développement-immunodéfi...,fr,None,ICD-10,E72.1,Code attribué (CIM-10/CIM-11: Le code cible es...,21604
8329,619972,Maladie CADINS,fr,Syndrome d'atopie avec interférence dominante ...,ICD-10,D81.8,Code attribué (CIM-10/CIM-11: Le code cible es...,21604
8330,619363,Syndrome NOCARH,fr,Syndrome de cytopénie-autoinflammation-éruptio...,ICD-10,D76.1,Code attribué (CIM-10/CIM-11: Le code cible es...,21604
8331,619360,NON RARE EN EUROPE : Persistance héréditaire i...,fr,None,ICD-10,D56.4,Code spécifique (CIM-10/CIM-11: Le code ORPHA ...,21583


In [26]:
df_icd[df_icd.code=="Q773"]

,code,aut_mco,pos,aut_ssr,lib_court,libelle,groupe
12439,Q773,0,OOO,0,CHONDRODYSPLASIE PONCTUEE,Chondrodysplasie ponctuée,Diagnostics


# OFS database

In [ ]:
df_ofs_master = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/MASTER.TXT", sep="¦",encoding="latin-1")
df_ofs_system = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/SYSTEM.TXT", sep="¦",encoding="latin-1")
df_ofs_desc = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/LIBELLE.TXT", sep="¦",encoding="latin-1", doublequote=False)
df_ofs_include = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/INCLUDE.TXT", sep="¦",encoding="latin-1", doublequote=False)
df_ofs_exclude = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/EXCLUDE.TXT", sep="¦",encoding="latin-1", doublequote=False)

/var/folders/ll/25thrhx56t34sp92v0d0hzdr0000gn/T/ipykernel_33202/2207474830.py:1: ParserWarning: Falling back to the 'python' engine because the separator encoded in utf-8 is > 1 char long, and the 'c' engine does not support such separators; you can avoid this warning by specifying engine='python'.
  df_ofs_master = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/MASTER.TXT", sep="¦",encoding="latin-1")
/var/folders/ll/25thrhx56t34sp92v0d0hzdr0000gn/T/ipykernel_33202/2207474830.py:2: ParserWarning: Falling back to the 'python' engine because the separator encoded in utf-8 is > 1 char long, and the 'c' engine does not support such separators; you can avoid this warning by specifying engine='python'.
  df_ofs_system = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/SYSTEM.TXT", sep="¦",encoding="latin-1")
/var/folders/ll/25thrhx56t34sp92v0d0hzdr0000gn/T/ipykernel_33202/2207474830.py:3: ParserWarning: Falling back to the 'python' engine because the separator encoded in utf-8 is > 1 char long, and the 'c

## Build a list of diseases from ICD entities

La CIM 10 permet différents niveaux de détail en fonction des pathologies, ce qui sous entend que chaque pathologie n'est pas décrites avec le même nombre de codes. Le niveau de base de la classification est la catégorie, mais au sein d'une catégorie le nombre de code est fixe, si bien qu'un même pathologie peut être décrite à l'aide de plusieurs catétogies formant des blocs de codes. A l'inverse des pathologie ne sont décrite que par un seul code au sein d'une catégorie résiduelle "autres". Le distinction entre les pathologies étant décrites avec de nobmre codes répartis en plusieurs catégories et celles décrite par un code ou non décrites en tant que telles, est dicté principalement par la notion de fréquence. Dans ce contexte, la hiérarchie en chapitre, bloc, catégorie ne permet pas en soi de disposer d'une liste de l'ensemble des pathologies décrites dans la CIM. Un travail spécifique visant à identifier le bon niveau de regroupement pour chaque pathologie est nécessaire.

On commence par le niveau bloc, sont exlcus tous les blocs mentionnant dans leur libellé le mots "autres". Pour les blocs autres exclus, on utilise le niveau hiérachique inférieur (sous-bloc). On procède ainsi en parcourant l'arbre pour chaque niveau de la hiérachique en excluant les libellés des entités qui contiennent le terme "autres" et qui sont remplacer par les entités du niveau inférieur.

On obtient ainsi une liste d'environ 500 pathologies.


In [46]:
df_u = df_ofs_master[df_ofs_master.type=="U"]
df_g =  df_ofs_master[(df_ofs_master.type=="G") & (~ df_ofs_master.SID.isin(df_u.id2))]

In [54]:
df_blocs = pd.concat([df_u,df_g])[["SID","code","sort","level"]].merge(df_ofs_system).merge(df_ofs_desc[["LID","libelle"]]).sort_values("sort")

In [83]:
sid_blocs_others = df_blocs.loc[df_blocs.libelle.str.contains("autres|certaines"),"SID"].to_list()

In [ ]:
df_categ = df_ofs_master[df_ofs_master.type=="K"]

In [66]:
df_u_ = df_u[~(df_u.SID.isin(sid_blocs_others))]
df_g_ = df_g[~(df_g.SID.isin(sid_blocs_others))]
df_categ_ = df_categ[(df_categ.id2.isin(sid_blocs_others)) | (df_categ.id3.isin(sid_blocs_others)) ]
df_tagets_level_1 = pd.concat([df_u_,df_g_,df_categ_])[["SID","code","sort","level"]].merge(df_ofs_system).merge(df_ofs_desc[["LID","libelle"]]).sort_values("sort")

In [85]:
level_1_others = df_tagets_level_1.loc[df_tagets_level_1.libelle.str.contains("autres|certaines"),"SID"].to_list() 

In [88]:
df_sub = df_ofs_master[df_ofs_master.type=="K"]

In [95]:
df_tagets_level_2 = pd.concat([df_tagets_level_1[~(df_tagets_level_1.SID.isin(level_1_others))],
          df_sub.loc[df_sub.id3.isin(level_1_others),["SID","code","sort","level"]]]).merge(df_ofs_system).merge(df_ofs_desc[["LID","libelle"]]).sort_values("sort")

In [53]:
from IPython.display import HTML

In [97]:
HTML(df_tagets_level_2.to_html())

,SID,code,sort,level,LID,libelle
0,2,(A00-A09),A00-,2,2.0,maladies intestinales infectieuses
1,71,(A15-A19),A15',2,71.0,tuberculose
2,115,A20,A20-,3,115.0,peste
3,123,A21,A21,3,123.0,tularémie
4,131,A22,A22,3,131.0,charbon
5,138,A23,A23,3,138.0,brucellose
6,145,A24,A24,3,145.0,morve et mélioïdose
7,151,A25,A25,3,151.0,fièvres causées par morsure de rat
8,155,A26,A26,3,155.0,érysipéloïde
9,160,A27,A27,3,160.0,leptospirose


### Les notes d'inclusion et d'exclusion 
Les notes di'nclusion et d'exclusions pourront être utiles pour la construction du modèle permettant d'identifier les codes


In [ ]:

df_ofs_include = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/INCLUDE.TXT", sep="¦",encoding="latin-1", doublequote=False)
df_ofs_exclude = pd.read_csv(PATH_DATA +"CIM_OFS_SW_2006/EXCLUDE.TXT", sep="¦",encoding="latin-1", doublequote=False)

In [104]:
df_ofs_master.loc[df_ofs_master.level>2,["SID","abbrev","sort"]].merge(df_ofs_include).merge(df_ofs_desc[["LID","libelle"]]).sort_values("sort")

,SID,abbrev,sort,LID,libelle
0,13,A02,A02,26794,infection ou intoxication alimentaire due à to...
1,45,A06,A06,26795,infection à Entamoeba histolytica
2,108,A19,A19,26797,tuberculose disséminée
3,108,A19,A19,26798,tuberculose généralisée
4,108,A19,A19,26799,polysérite tuberculeuse
...,...,...,...,...,...
1221,11914,Z49,Z49,27324,préparation d'une dialyse et traitement
1222,12218,Z89,Z89,27326,perte d'un membre post-traumatique
1223,12218,Z89,Z89,27327,perte d'un membre après intervention chirurgicale
1224,12229,Z90,Z90,27328,perte d'une partie du corps NCA après interven...


In [107]:
df_ofs_master.loc[df_ofs_master.level>2,["SID","abbrev","level","type","sort"]].merge(df_ofs_exclude).merge(df_ofs_desc[["LID","libelle"]]).sort_values("sort")

,SID,abbrev,level,type,sort,excl,plus,LID,daget,libelle
0,26,A04,3,K,A04,102,0,28852,NaN,entérite tuberculeuse
1,26,A04,3,K,A04,37,1,29456,NaN,intoxications bactériennes d'origine alimentaire
2,33,A046,4,S,A04.6,167,0,31150,NaN,yersiniose extra-intestinale
11,37,A05,3,K,A05,185,1,29617,NaN,listériose
9,37,A05,3,K,A05,9808,0,28768,NaN,effets toxiques de denrées alimentaires nocives
...,...,...,...,...,...,...,...,...,...,...
5518,12306,Z98,3,K,Z98,11929,0,30608,NaN,soins de contrôle médicaux et de convalescence
5508,12306,Z98,3,K,Z98,11954,1,30608,NaN,soins de contrôle médicaux et de convalescence
5510,12306,Z98,3,K,Z98,11870,0,30608,NaN,soins de contrôle médicaux et de convalescence
5513,12306,Z98,3,K,Z98,11895,0,30608,NaN,soins de contrôle médicaux et de convalescence


In [106]:
df_ofs_master

,SID,code,sort,abbrev,level,type,id1,id2,id3,id4,id5,id6,id7,valid,date,author,comment
0,1,(A00-B99),A00',(A00-B99),1,C,1,0,0,0,0,0,0,1,14.7.2003 00:00:00,start,
1,2,(A00-A09),A00-,(A00-A09),2,G,1,2,0,0,0,0,0,1,14.7.2003 00:00:00,start,
2,3,A00,"A00,",A00,3,K,1,2,3,0,0,0,0,1,14.7.2003 00:00:00,start,
3,4,A00.0,A00.0,A000,4,S,1,2,3,4,0,0,0,1,14.7.2003 00:00:00,start,
4,5,A00.1,A00.1,A001,4,S,1,2,3,5,0,0,0,1,14.7.2003 00:00:00,start,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19150,19547,F66.92,F66.92,F6692,5,D,2422,2742,2783,2788,19547,0,0,1,14.7.2003 00:00:00,start,
19151,19548,F66.98,F66.98,F6698,5,D,2422,2742,2783,2788,19548,0,0,1,14.7.2003 00:00:00,start,
19152,19549,T14.20,T14.20,T1420,5,D,8571,9340,9374,9377,19549,0,0,1,14.7.2003 00:00:00,start,
19153,19550,T14.21,T14.21,T1421,5,D,8571,9340,9374,9377,19550,0,0,1,14.7.2003 00:00:00,start,
